In [1]:
# external
import numpy as np
import healpy as hp
import matplotlib.pyplot as plt
# modules from cmblensplus
import cmblensplus.basic as basic
import cmblensplus.curvedsky as cs
import constant as c
import cmb
import time
from tqdm import trange

In [2]:
import ducc0

Set parameters

In [3]:
nside = 2048  # Nside of Healpix map
npix  = 12*nside**2  # Numer of pixels of Healpix map
lmax  = 2*nside # Maximum multipole of harmonic coefficients to be computed

In [4]:
# ucl is an array of shape [0:5,0:rlmax+1] and ucl[0,:] = TT, ucl[1,:] = EE, ucl[2,:] = TE, lcl[3,:] = phiphi, lcl[4,:] = Tphi
ucl = cmb.read_camb_cls('../data/unlensedcls.dat',ftype='scal',output='array')[:,:lmax+1]

Generate random gaussian fields

In [5]:
Talm = cs.utils.gauss1alm(ucl[0,:])

In [6]:
alm = hp.synalm(ucl[0,:], lmax=lmax, new=True)

In [7]:
hb = ducc0.healpix.Healpix_Base(nside, "RING")
mp = np.empty((1, hb.npix()), dtype=np.float64)

In [8]:
times = []
for _ in trange(20, desc="ducc0 synthesis"):
    t0 = time.perf_counter()
    __ = cs.utils.hp_alm2map(nside,Talm)
    times.append(time.perf_counter() - t0)
print("median:", np.median(times))
print("mean:  ", np.mean(times))
print("min:   ", np.min(times))

ducc0 synthesis: 100%|██████████| 20/20 [01:57<00:00,  5.86s/it]

median: 5.813864752999507
mean:   5.862399006792112
min:    5.752301453961991


In [9]:
times = []
for _ in trange(20, desc="ducc0 synthesis"):
    t0 = time.perf_counter()
    ducc0.sht.synthesis(alm=alm.reshape(1, -1),map=mp,spin=0,lmax=lmax,mmax=lmax,nthreads=2,**hb.sht_info())
    times.append(time.perf_counter() - t0)
print("median:", np.median(times))
print("mean:  ", np.mean(times))
print("min:   ", np.min(times))

ducc0 synthesis: 100%|██████████| 20/20 [01:46<00:00,  5.32s/it]

median: 5.2448146670358256
mean:   5.316234588692896
min:    5.135075814090669


In [10]:
times = []
for _ in trange(20, desc="ducc0 synthesis"):
    t0 = time.perf_counter()
    ducc0.sht.synthesis(alm=alm.reshape(1, -1),map=mp,spin=0,lmax=lmax,mmax=lmax,nthreads=8,**hb.sht_info())
    times.append(time.perf_counter() - t0)
print("median:", np.median(times))
print("mean:  ", np.mean(times))
print("min:   ", np.min(times))

ducc0 synthesis: 100%|██████████| 20/20 [01:45<00:00,  5.28s/it]

median: 5.266640629037283
mean:   5.283595453464659
min:    5.194421743042767


Lower resolution

In [11]:
nside = 1024  # Nside of Healpix map
npix  = 12*nside**2  # Numer of pixels of Healpix map
lmax  = 2*nside # Maximum multipole of harmonic coefficients to be computed

In [12]:
hb = ducc0.healpix.Healpix_Base(nside, "RING")
mp = np.empty((1, hb.npix()), dtype=np.float64)

In [13]:
times = []
for _ in trange(20, desc="ducc0 synthesis"):
    t0 = time.perf_counter()
    __ = cs.utils.hp_alm2map(nside,Talm[:lmax+1,:lmax+1])
    times.append(time.perf_counter() - t0)
print("median:", np.median(times))
print("mean:  ", np.mean(times))
print("min:   ", np.min(times))

ducc0 synthesis: 100%|██████████| 20/20 [00:18<00:00,  1.09it/s]

median: 0.9160314564942382
mean:   0.9171140098012984
min:    0.8959196800133213


In [14]:
times = []
for _ in trange(20, desc="ducc0 synthesis"):
    t0 = time.perf_counter()
    ducc0.sht.synthesis(alm=alm.reshape(1, -1),map=mp,spin=0,lmax=lmax,mmax=lmax,nthreads=2,**hb.sht_info())
    times.append(time.perf_counter() - t0)
print("median:", np.median(times))
print("mean:  ", np.mean(times))
print("min:   ", np.min(times))

ducc0 synthesis: 100%|██████████| 20/20 [00:15<00:00,  1.32it/s]

median: 0.7516852514818311
mean:   0.7553168847109191
min:    0.7339801890775561
